##Importing Dependencies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

df = pd.read_csv('Bank_Personal_Loan_Modelling.csv')
df.head()

##Data Quality Check

In [ ]:
print(df.shape)
print(df.info())
print('Duplicates:', df.duplicated().sum())
print(df.isna().sum())
print(df.describe().T.round(2))

##Feature Engineering

In [ ]:
df.columns = [c.strip().lower().replace(' ','_') for c in df.columns]
df['education_label'] = df['education'].map({1:'Undergraduate',2:'Graduate',3:'Advanced/Professional'}).fillna('Unknown')
df['income_segment'] = pd.cut(df['income'], [-np.inf,49,99,np.inf], labels=['Low','Middle','High'])
df['age_segment'] = pd.cut(df['age'], [-np.inf,29,39,49,np.inf], labels=['Young','Early Career','Mid Career','Senior'])
df['digital_engaged'] = ((df['online']==1) | (df['creditcard']==1)).astype(int)
df['relationship_depth'] = df[['securities_account','cd_account','online','creditcard']].sum(axis=1)
df[['income_segment','age_segment','digital_engaged','relationship_depth']].head()

##Business EDA

In [ ]:
print('Overall loan conversion:', round(df['personal_loan'].mean()*100,2),'%')
print('Income segment conversion')
print((df.groupby('income_segment', observed=False)['personal_loan'].mean()*100).round(2))
print('Digital banking conversion')
print((df.groupby('online')['personal_loan'].mean()*100).round(2))
print('Education conversion')
print((df.groupby('education_label')['personal_loan'].mean()*100).round(2))

In [ ]:
plt.figure(figsize=(8,5))
sns.barplot(data=df, x='income_segment', y='personal_loan', estimator=np.mean)
plt.ylabel('Loan conversion rate')
plt.title('Personal-loan conversion by income segment')
plt.show()

##Model Preparation

In [ ]:
features=['age','experience','income','family','ccavg','education','mortgage','securities_account','cd_account','online','creditcard']
target='personal_loan'
X=df[features]
y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
print('Train:', X_train.shape, 'Test:', X_test.shape)

##logistic Regression baseline

In [ ]:
logit=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scale',StandardScaler()),
    ('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42))
])
logit.fit(X_train,y_train)
p_log=logit.predict_proba(X_test)[:,1]
pred_log=(p_log>=0.5).astype(int)
print(classification_report(y_test,pred_log))
print('ROC-AUC:', round(roc_auc_score(y_test,p_log),4))

##Random Forest Model

In [ ]:
rf=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('model',RandomForestClassifier(n_estimators=500,max_depth=8,min_samples_leaf=5,class_weight='balanced',random_state=42,n_jobs=-1))
])
rf.fit(X_train,y_train)
p_rf=rf.predict_proba(X_test)[:,1]
pred_rf=(p_rf>=0.5).astype(int)
metrics = {
    'accuracy':accuracy_score(y_test,pred_rf),
    'precision':precision_score(y_test,pred_rf),
    'recall':recall_score(y_test,pred_rf),
    'f1':f1_score(y_test,pred_rf),
    'roc_auc':roc_auc_score(y_test,p_rf)
}
print({k:round(v,4) for k,v in metrics.items()})
ConfusionMatrixDisplay.from_predictions(y_test,pred_rf)
plt.title('Random Forest confusion matrix')
plt.show()

##Feature Importance

In [ ]:
importance = pd.DataFrame({'feature':features,'importance':rf.named_steps['model'].feature_importances_}).sort_values('importance',ascending=False)
print(importance)
sns.barplot(data=importance.head(10), y='feature', x='importance')
plt.title('Top model features')
plt.show()

##Score all customers and create campaign tiers

In [ ]:
rf.fit(X,y)
df['loan_propensity']=rf.predict_proba(X)[:,1]
df['propensity_score']=(df['loan_propensity']*100).round(1)
df['priority_tier']=pd.cut(df['propensity_score'],[-np.inf,39.999,59.999,79.999,np.inf],labels=['Low','C','B','A'])
income_norm=(df['income']-df['income'].min())/(df['income'].max()-df['income'].min())
rel_norm=df['relationship_depth']/4
df['opportunity_score']=(100*(0.60*df['loan_propensity']+0.25*income_norm+0.15*rel_norm)).round(1)
df['campaign_priority']=np.where((df['personal_loan']==0)&(df['propensity_score']>=60),'Target','Do not target')
df['recommended_channel']=np.select([
    (df['personal_loan']==0)&(df['opportunity_score']>=70)&(df['online']==1),
    (df['personal_loan']==0)&(df['opportunity_score']>=60),
    (df['personal_loan']==0)&(df['opportunity_score']>=45)
],['Digital / pre-approved offer','Relationship Manager','Branch / assisted campaign'],default='No campaign')
df.sort_values('opportunity_score',ascending=False).head(20)

In [ ]:
df.to_csv('customer_scored.csv',index=False)
print('Saved customer_scored.csv')